In [122]:
import emoji, re, string, time
import pandas as pd
import numpy as np
from scipy.stats import randint

import nltk
from nltk.corpus import stopwords
import spacy

from sklearn.feature_extraction.text import CountVectorizer


In [123]:
unicode_emoji = {}
for key, value in emoji.EMOJI_DATA.items():
    try:
        unicode_emoji[key] = value['pt']
    except:
        pass

#emojis and punctuation
emojis_list = list(unicode_emoji)
punct = list(string.punctuation)
emojis_punct = emojis_list + punct

def processEmojisPunctuation(text, remove_punct = True):
    '''
    Put spaces between emojis. Removes punctuation.
    '''
    #get all unique chars
    chars = set(text)
    #for each unique char in text, do:
    for c in chars:
        #remove punctuation
        if remove_punct:
            if c in emojis_list:
                text = text.replace(c, ' ' + c + ' ')
            if c in punct:
                text = text.replace(c, ' ')

        #put spaces between punctuation
        else:
            if c in emojis_punct:
                text = text.replace(c, ' ' + c + ' ')          

    text = text.replace('  ', ' ')
    return text

#stop words removal
stop_words = list(stopwords.words('portuguese'))
new_stopwords = ['aí','pra','vão','vou','onde','lá','aqui',
                'tá','pode','pois','so','deu','agora','todo',
                'nao','ja','vc', 'bom', 'ai','kkk','kkkk','ta', 'voce', 'alguem', 'ne', 'pq',
                'cara','to','mim','la','vcs','tbm', 'tudo', 'ir', 'face', 'index', 'hands', 'pointing',
                'keycap', 'backhand', 'floor', 'laughing', 'on', 'the', 'rolling', 'skin', 'tone', 'keycap',
                'folded', 'light', 'up', 'with', 'car', 'of', 'joy', 'police', 'data', 'clapping', 
                'down', 'de', 'esse', 'tear', 'mark', 'check', 'heavy', 'dólar', 'sign', 'button', 'ball', 
                'soccer', 'em', 'médium', 'dark', 'warning', 'double', 'exclamation', 'smiling', 'eyes',
                'fluttering', 'in', 'wind', 'raising', 'dia', 'fear' , 'screaming' , 'heart', 'red', 'tarde', 'noite'   ]
stop_words = stop_words + new_stopwords
final_stop_words = []
for sw in stop_words:
    sw = ' '+ sw + ' '
    final_stop_words.append(sw)

def removeStopwords(text):
    for sw in final_stop_words:
        text = text.replace(sw,' ')
    text = text.replace('  ',' ')
    return text

#lemmatization
nlp = spacy.load('pt_core_news_sm')
def lemmatization(text):
    doc = nlp(text)
    for token in doc:
        if token.text != token.lemma_:
            text = text.replace(token.text, token.lemma_)
    return text


def domainUrl(text):
    '''
    Substitutes an URL in a text for the domain of this URL
    Input: an string
    Output: the string with the modified URL
    '''    
    if 'http' in text:
        re_url = '[^\s]*https*://[^\s]*'
        matches = re.findall(re_url, text, flags=re.IGNORECASE)
        for m in matches:
            domain = m.split('//')
            domain = domain[1].split('/')[0]
            text = re.sub(re_url, domain, text, 1)
        return text
    else:
        return text 

def preprocess(text):
    text = text.lower().strip()
    text = domainUrl(text)
    text = processEmojisPunctuation(text)
    text = removeStopwords(text)
    text = lemmatization(text)
    return text


In [124]:
# Função de pré-processamento
def preprocess_data(df, experiment):
    if 'processed' in experiment:
        print("Pré-processamento ativado.")
        pro_texts = [preprocess(t) for t in df['text_content_anonymous']]
    else:
        print("Sem pré-processamento.")
        pro_texts = [processEmojisPunctuation(t.lower(), remove_punct=False) for t in df['text_content_anonymous']]
    return pro_texts


## WhatsApp

In [125]:
df = pd.read_csv('./Datasets/Correto_whatsapp_rotulado_revisado.csv')
df.head()

,text_content_anonymous,preconceito
0,Show :clapping_hands_light_skin_tone::clapping...,0
1,Mais uma vez o PT= Partido das Trevas apela ao...,1
2,Se não puder avise só para não quebrar a vigíl...,0
3,E a esquerda tenta impedir o trabalho,0
4,Deus perdoa porque eles Não sabem que fazem e ...,0


In [126]:
# === Pré-processamento dinâmico ===
df["text"] = preprocess_data(df, experiment="processed")  
df.head()

Pré-processamento ativado.


,text_content_anonymous,preconceito,text
0,Show :clapping_hands_light_skin_tone::clapping...,0,show clapping clapping nordestino acordar
1,Mais uma vez o PT= Partido das Trevas apela ao...,1,mais vez pt partir treva apelar mp proibir esc...
2,Se não puder avise só para não quebrar a vigíl...,0,se puder aviserr quebrar vigíliar \n\n“pai est...
3,E a esquerda tenta impedir o trabalho,0,e esquerda tentar impedir trabalho
4,Deus perdoa porque eles Não sabem que fazem e ...,0,Deus perdoa porque saber fazer falar pronto as...


In [127]:
stopwords_custom = [
    'aí','pra','vão','vou','onde','lá','aqui','tá','pode','pois','so','deu','agora','todo',
    'nao','ja','vc', 'bom', 'ai','kkk','kkkk','ta', 'voce', 'alguem', 'ne', 'pq','cara','to','mim',
    'la','vcs','tbm', 'tudo', 'ir', 'face', 'index', 'hands', 'pointing','keycap', 'backhand',
    'floor', 'laughing', 'on', 'the', 'rolling', 'skin', 'tone', 'folded', 'light', 'up', 'with',
    'car', 'of', 'joy', 'police', 'data', 'clapping', 'down', 'de', 'esse', 'tear', 'mark', 'check',
    'heavy', 'dólar', 'sign', 'button', 'ball', 'soccer', 'em', 'médium', 'dark', 'warning',
    'double', 'exclamation', 'smiling', 'eyes','fluttering', 'in', 'wind', 'raising', 'dia',
    'fear', 'screaming', 'heart', 'red', 'tarde', 'noite', 'thinking', 'kitchen', 'knife', 'winking', 
    'pinched', 'fingers', 'oncoming', 'facepalming', 'man', 'thumbs', 'enraged', 'fist', 'cross', 'green', 
    'yellow', 'prohibited' , 'steam', 'from', 'nose', 'crossbone', 'dagger'
]



def remove_custom_words(text, remove_list):
    tokens = re.findall(r'\w+', text.lower(), flags=re.UNICODE)
    tokens = [t for t in tokens if t not in remove_list]
    return " ".join(tokens)

df['clean_text'] = df['text'].astype(str).apply(
    lambda t: remove_custom_words(t, stopwords_custom)
)


In [128]:
# Separar em duas classes
df_pos = df[df['preconceito'] == 1]
df_neg = df[df['preconceito'] == 0]

texts_pos = df_pos['clean_text'].astype(str).tolist()
texts_neg = df_neg['clean_text'].astype(str).tolist()


In [129]:
def top_ngrams(texts, n=1, top_k=10):
    vec = CountVectorizer(ngram_range=(n, n), min_df=2)
    X = vec.fit_transform(texts)
    freqs = zip(vec.get_feature_names_out(), X.sum(axis=0).A1)
    sorted_ngrams = sorted(freqs, key=lambda x: x[1], reverse=True)
    return sorted_ngrams[:top_k], set(vec.get_feature_names_out())


In [130]:
# unigramas
top_uni_pos, set_uni_pos = top_ngrams(texts_pos, n=1)
top_uni_neg, set_uni_neg = top_ngrams(texts_neg, n=1)

# bigramas
top_bi_pos, set_bi_pos = top_ngrams(texts_pos, n=2)
top_bi_neg, set_bi_neg = top_ngrams(texts_neg, n=2)

# trigramas
top_tri_pos, set_tri_pos = top_ngrams(texts_pos, n=3)
top_tri_neg, set_tri_neg = top_ngrams(texts_neg, n=3)


In [131]:
# palavras exclusivas 
exclusive_uni_pos = set_uni_pos - set_uni_neg
exclusive_uni_neg = set_uni_neg - set_uni_pos

exclusive_bi_pos = set_bi_pos - set_bi_neg
exclusive_bi_neg = set_bi_neg - set_bi_pos

exclusive_tri_pos = set_tri_pos - set_tri_neg
exclusive_tri_neg = set_tri_neg - set_tri_pos


In [132]:
print("=== TOP 10 Unigramas – PRECONCEITO ===")
print(top_uni_pos)
print("\n=== TOP 10 Unigramas – NÃO PRECONCEITO ===")
print(top_uni_neg)

print("\n=== TOP 10 Bigramas – PRECONCEITO ===")
print(top_bi_pos)
print("\n=== TOP 10 Bigramas – NÃO PRECONCEITO ===")
print(top_bi_neg)

print("\n=== TOP 10 Trigramas – PRECONCEITO ===")
print(top_tri_pos)
print("\n=== TOP 10 Trigramas – NÃO PRECONCEITO ===")
print(top_tri_neg)

print("\n=== Exclusivos (Unigramas) – PRECONCEITO ===")
print(list(exclusive_uni_pos)[:20])

print("\n=== Exclusivos (Unigramas) – NÃO PRECONCEITO ===")
print(list(exclusive_uni_neg)[:20])

print("\n=== Exclusivos (Bigramas) – PRECONCEITO ===")
print(list(exclusive_bi_pos)[:20])

print("\n=== Exclusivos (Bigramas) – NÃO PRECONCEITO ===")
print(list(exclusive_bi_neg)[:20])

print("\n=== Exclusivos (Trigrams) – PRECONCEITO ===")
print(list(exclusive_tri_pos)[:20])

print("\n=== Exclusivos (Trigrams) – NÃO PRECONCEITO ===")
print(list(exclusive_tri_neg)[:20])


=== TOP 10 Unigramas – PRECONCEITO ===
[('brazil', 300), ('luladrão', 276), ('fazer', 237), ('bolsonaro', 228), ('lula', 212), ('votar', 188), ('brasil', 167), ('luladrao', 163), ('falar', 159), ('idiota', 157)]

=== TOP 10 Unigramas – NÃO PRECONCEITO ===
[('brazil', 848), ('deus', 745), ('bolsonaro', 339), ('presidente', 230), ('brasil', 221), ('fazer', 215), ('votar', 178), ('medium', 157), ('jesus', 151), ('senhor', 141)]

=== TOP 10 Bigramas – PRECONCEITO ===
[('brazil brazil', 185), ('medium medium', 24), ('primeiro turno', 23), ('dollar dollar', 22), ('povo brasileiro', 22), ('votar lula', 21), ('presidente bolsonaro', 20), ('ex presidiário', 19), ('pt luladrão', 18), ('vomiting vomiting', 18)]

=== TOP 10 Bigramas – NÃO PRECONCEITO ===
[('brazil brazil', 492), ('deputado federal', 58), ('deus abençoe', 51), ('deputado estadual', 50), ('presidente bolsonaro', 44), ('deus pátria', 40), ('deus acima', 37), ('bolsonaro 22', 36), ('jair bolsonaro', 35), ('nome jesus', 34)]

=== TOP 1

## Telegram

In [133]:
df_tel = pd.read_csv('./Datasets/Telegram_Tratado_Rotulado_Revisado_Final.csv')
df_tel.head()

,text_content_anonymous,preconceito
0,"Sim, eu sou um ""decepcionado""!\n\nEu vejo pess...",0
1,"Em épocas com muitos eventos grandes, altas qu...",0
2,Algumas premissas que ele adotou:\n- Ser de di...,0
3,Apocalipse 22:11. Cristo fez expiação por Seu ...,0
4,:sun_with_face:S:rose:H:cherry_blossom:A:sunfl...,0


In [134]:
# === Pré-processamento dinâmico ===
df_tel["text"] = preprocess_data(df_tel, experiment="processed")  
df_tel.head()



Pré-processamento ativado.


,text_content_anonymous,preconceito,text
0,"Sim, eu sou um ""decepcionado""!\n\nEu vejo pess...",0,sim decepcionar \n\neu vejo pessoa tentar jeit...
1,"Em épocas com muitos eventos grandes, altas qu...",0,em época muito evento grande alta quantidade e...
2,Algumas premissas que ele adotou:\n- Ser de di...,0,algum premissa adotar \n direita \n ter rede s...
3,Apocalipse 22:11. Cristo fez expiação por Seu ...,0,apocalipse 22 11 cristo fazer expiação povo a...
4,:sun_with_face:S:rose:H:cherry_blossom:A:sunfl...,0,sun s rose h cherry blossom sunflower l hibis...


In [135]:
stopwords_custom = [
    'aí','pra','vão','vou','onde','lá','aqui','tá','pode','pois','so','deu','agora','todo',
    'nao','ja','vc', 'bom', 'ai','kkk','kkkk','ta', 'voce', 'alguem', 'ne', 'pq','cara','to','mim',
    'la','vcs','tbm', 'tudo', 'ir', 'face', 'index', 'hands', 'pointing','keycap', 'backhand',
    'floor', 'laughing', 'on', 'the', 'rolling', 'skin', 'tone', 'folded', 'light', 'up', 'with',
    'car', 'of', 'joy', 'police', 'data', 'clapping', 'down', 'de', 'esse', 'tear', 'mark', 'check',
    'heavy', 'dólar', 'sign', 'button', 'ball', 'soccer', 'em', 'médium', 'dark', 'warning',
    'double', 'exclamation', 'smiling', 'eyes','fluttering', 'in', 'wind', 'raising', 'dia',
    'fear', 'screaming', 'heart', 'red', 'tarde', 'noite', 'thinking', 'kitchen', 'knife', 'winking', 
    'pinched', 'fingers', 'oncoming', 'facepalming', 'man', 'thumbs', 'enraged', 'fist', 'cross', 'green', 
    'yellow', 'prohibited' , 'steam', 'from', 'nose', 'crossbone', 'dagger'
]


def remove_custom_words(text, remove_list):
    tokens = re.findall(r'\w+', text.lower(), flags=re.UNICODE)
    tokens = [t for t in tokens if t not in remove_list]
    return " ".join(tokens)

df_tel['clean_text'] = df_tel['text'].astype(str).apply(
    lambda t: remove_custom_words(t, stopwords_custom)
)


In [136]:
# Separar em duas classes
df_tel_pos = df_tel[df_tel['preconceito'] == 1]
df_tel_neg = df_tel[df_tel['preconceito'] == 0]

texts_pos_tel = df_tel_pos['clean_text'].astype(str).tolist()
texts_neg_tel = df_tel_neg['clean_text'].astype(str).tolist()


In [137]:
# unigramas
top_uni_pos_tel, set_uni_pos_tel = top_ngrams(texts_pos_tel, n=1)
top_uni_neg_tel, set_uni_neg_tel = top_ngrams(texts_neg_tel, n=1)

# bigramas
top_bi_pos_tel, set_bi_pos_tel = top_ngrams(texts_pos_tel, n=2)
top_bi_neg_tel, set_bi_neg_tel = top_ngrams(texts_neg_tel, n=2)

# trigramas
top_tri_pos_tel, set_tri_pos_tel = top_ngrams(texts_pos_tel, n=3)
top_tri_neg_tel, set_tri_neg_tel = top_ngrams(texts_neg_tel, n=3)


In [138]:
# palavras exclusivas 
exclusive_uni_pos_tel = set_uni_pos_tel - set_uni_neg_tel
exclusive_uni_neg_tel = set_uni_neg_tel - set_uni_pos_tel

exclusive_bi_pos_tel = set_bi_pos_tel - set_bi_neg_tel
exclusive_bi_neg_tel = set_bi_neg_tel - set_bi_pos_tel

exclusive_tri_pos_tel = set_tri_pos_tel - set_tri_neg_tel
exclusive_tri_neg_tel = set_tri_neg_tel - set_tri_pos_tel


In [139]:
print("=== TOP 10 Unigramas – PRECONCEITO ===")
print(top_uni_pos_tel)
print("\n=== TOP 10 Unigramas – NÃO PRECONCEITO ===")
print(top_uni_neg_tel)

print("\n=== TOP 10 Bigramas – PRECONCEITO ===")
print(top_bi_pos_tel)
print("\n=== TOP 10 Bigramas – NÃO PRECONCEITO ===")
print(top_bi_neg_tel)

print("\n=== TOP 10 Trigramas – PRECONCEITO ===")
print(top_tri_pos_tel)
print("\n=== TOP 10 Trigramas – NÃO PRECONCEITO ===")
print(top_tri_neg_tel)

print("\n=== Exclusivos (Unigramas) – PRECONCEITO ===")
print(list(exclusive_uni_pos_tel)[:20])

print("\n=== Exclusivos (Unigramas) – NÃO PRECONCEITO ===")
print(list(exclusive_uni_neg_tel)[:20])

print("\n=== Exclusivos (Bigramas) – PRECONCEITO ===")
print(list(exclusive_bi_pos_tel)[:20])

print("\n=== Exclusivos (Bigramas) – NÃO PRECONCEITO ===")
print(list(exclusive_bi_neg_tel)[:20])

print("\n=== Exclusivos (Trigrams) – PRECONCEITO ===")
print(list(exclusive_tri_pos_tel)[:20])

print("\n=== Exclusivos (Trigrams) – NÃO PRECONCEITO ===")
print(list(exclusive_tri_neg_tel)[:20])


=== TOP 10 Unigramas – PRECONCEITO ===
[('brazil', 406), ('bolsonaro', 397), ('deus', 295), ('fazer', 284), ('comunista', 266), ('presidente', 256), ('povo', 231), ('ladrão', 225), ('brasil', 208), ('lula', 187)]

=== TOP 10 Unigramas – NÃO PRECONCEITO ===
[('deus', 822), ('bolsonaro', 387), ('brazil', 339), ('fazer', 315), ('presidente', 270), ('votar', 260), ('senhor', 249), ('povo', 228), ('dizer', 213), ('grupo', 211)]

=== TOP 10 Bigramas – PRECONCEITO ===
[('brazil brazil', 236), ('presidente bolsonaro', 51), ('flexed bicep', 45), ('ex presidiário', 25), ('brozil brozil', 23), ('segundo turno', 23), ('medium medium', 20), ('povo brasileiro', 20), ('fazer nada', 19), ('primeiro turno', 19)]

=== TOP 10 Bigramas – NÃO PRECONCEITO ===
[('brazil brazil', 151), ('medium medium', 67), ('daniel silveira', 55), ('jesus cristo', 39), ('deus abençoe', 36), ('estar escrito', 35), ('presidente bolsonaro', 35), ('segundo turno', 35), ('open book', 33), ('bolsonaro 22', 32)]

=== TOP 10 Trigra